In [ ]:
# run this once to install the required packages
%pip install pandas==2.3.3
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy
%pip install openpyxl

# DynaTMT (Current version: 2.9.3 - 2026-03-18)
%pip install --upgrade "git+https://github.com/science64/DynaTMT.git"

# Current version: 2.1.3 (2026-02-02)
%pip install --upgrade "git+https://github.com/science64/PBLMM.git"

In [1]:
# import the required packages / you need to run all the time!
from datetime import date
import warnings
import DynaTMT.DynaTMT as mePROD # should be ==> v2.9.3
import pandas as pd              # should be ==> v2.3.3
import PBLMM                     # should be ==> v2.1.3

warnings.filterwarnings("ignore")

# Check versions
print("╔════════════════════════════════════════╗")
print("║       Package Versions Loaded          ║")
print("╠════════════════════════════════════════╣")
print(f"║  DynaTMT:        {mePROD.__version__:<20}  ║")
print(f"║  Pandas:         v{pd.__version__:<20} ║")
print(f"║  PBLMM:          v{PBLMM.__version__:<20} ║")
print("╚════════════════════════════════════════╝")

╔════════════════════════════════════════╗
║       Package Versions Loaded          ║
╠════════════════════════════════════════╣
║  DynaTMT:        v2.9.3                ║
║  Pandas:         v2.3.3                ║
║  PBLMM:          v2.1.3                ║
╚════════════════════════════════════════╝


In [2]:
wd = "Example data/MS3_data" # you can define your folder here etc: C://Users/User/Data/fractionation/

nameOfStudy = "48h_siUSP39_DB" # please define a name for your study

dataName = "20240109_LU_LC2_MAA_DB_mePROD_MS3_PSMs.txt" # please define the name of your data file (PSMs) here

conditions = ['Cont', 'Cont', 'Cont', 'USP39', 'USP39', 'USP39'] # define the conditions of TMT multiplexing here
pairs = [['USP39', 'Cont']] # define the pairs of conditions you want to compare here. result will be log2(USP39/Cont)

In [3]:
psms = pd.read_csv(f'{wd}/{dataName}', sep='\t', header=0) # TEXT or CSV file: you provide your .txt PSM or peptide file here.

boster_removed = psms.drop('Abundance 131C', axis = True) # remove the booster channel if present  (location of the booster Abundance: 131C channel in the data file)
baseline_removed = boster_removed.drop('Abundance 126', axis = True) # remove the baseline channel (location of the baseline Abundance: 126 channel in the data file)

In [4]:
process = mePROD.PD_input(baseline_removed) # initiate your date here with PD_input class, if your data name is 'baseline_removed'

filter_data = process.filter_PSMs(baseline_removed) # filter contamination, NA samples, shared peptides

sumNorm = process.total_intensity_normalisation(filter_data) # for total intenstiy normalization

heavy = process.extract_heavy(sumNorm) # extract heavy PSMs/peptides

# light = process.extract_light(sumNorm) # extract light PSMs/peptides (OPTIONAL) ==> You need to export it later with peptide_to_protein conversion

peptide_data = process.PSMs_to_Peptide(heavy) # convert PSMs to Peptide level data

# PBLMM analysis ==> this is the main part of the statistical analysis based on peptide based linear mixed model (LMM)
hypothesis_testing = PBLMM.HypothesisTesting()
resultFinal = hypothesis_testing.peptide_based_lmm(peptide_data,conditions=conditions,pairs=pairs)
resultFinal.reset_index(inplace=True)
resultFinal.rename(columns={'index': 'Accession'}, inplace=True)

resultFinal.to_excel(f'{wd}/{nameOfStudy}_mePROD_PBLMM_MS3_{date.today().strftime("%d.%m.%Y")}.xlsx', index=False, engine='openpyxl')

print('[#] COMPLETED: resultFinal: %s rows x %s columns' % (resultFinal.shape[0], resultFinal.shape[1]))

Calling function: filter_PSMs
Calling function: total_intensity_normalisation
Calling function: extract_heavy
Extraction Done Extracted Heavy PSMs/Peptides: 22928
Calling function: PSMs_to_Peptide
Calculate Protein quantifications from Peptides
Combination done
Total Number of Datapoints:  89886
['USP39', 'Cont'] is analysing via PBLMM...
[#] COMPLETED: resultFinal: 3615 rows x 10 columns
